
# Demo: Create AI agent tools using Unity Catalog functions

[Create AI agent tools using Unity Catalog functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool)

Use Unity Catalog functions to create AI agent tools that execute custom logic and perform specific tasks that extend the capabilities of LLMs beyond language generation.

Unity Catalog tools are really just [user-defined functions (UDFs) in Unity Catalog](https://docs.databricks.com/aws/en/udf/unity-catalog) under the hood:

* `create_python_function` accepts a Python callable.
* `create_function` accepts a SQL body create function statement (incl. [Python functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-python-functions)).

## Requirements

🚨 NOTE: [Requirements](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#requirements) are not clear what the requirements are for creating and using UC functions. There are three paragraphs.

* Databricks Runtime 15.0 or later
* Python 3.10 or later
* [Serverless compute](https://docs.databricks.com/aws/en/compute/serverless/#requirements) must be enabled to execute Unity Catalog functions as AI agent tools in production.

## Unity Catalog functions vs. MCP servers

[When to use Unity Catalog functions vs. MCP servers](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#when-to-use-unity-catalog-functions-vs-mcp-servers)

In most cases, Databricks recommends MCP servers or defining the logic directly in agent code.

Databricks recommends using MCP servers to add Unity Catalog functions to your agent.

Use Unity Catalog functions as agent tools specifically for structured data retrieval tools when the query is known ahead of time and the agent provides the parameters.

## Create Unity Catalog Function

In [0]:
# Install Unity Catalog AI integration packages with the Databricks extras
# The Unity Catalog function integration is in the langchain-community package.
%pip install -U unitycatalog-ai[databricks]>=0.4.0 mlflow-skinny[databricks]>=3.12.0 databricks-agents>=1.10.2 databricks-mcp>=0.9.0 databricks-langchain>=0.19.0 langchain-community langchain
%restart_python

In [0]:
%pip show unitycatalog-ai

In [0]:
%pip show databricks-mcp

In [0]:
%pip show mlflow-skinny

In [0]:
%pip show databricks-agents

In [0]:
import mlflow

mlflow.autolog()


[Databricks Function Client](https://docs.unitycatalog.io/ai/client/#databricks-function-client)

`DatabricksFunctionClient` is a core component of the **Unity Catalog AI Core Library** to interact with Unity Catalog (UC) functions on Databricks.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

In [0]:
def add_numbers(number_1: float, number_2: float) -> float:
  """
  A function that accepts two floating point numbers adds them,
  and returns the resulting sum as a float.

  Args:
    number_1 (float): The first of the two numbers to add.
    number_2 (float): The second of the two numbers to add.

  Returns:
    float: The sum of the two input numbers.
  """
  return number_1 + number_2

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_agents;
CREATE SCHEMA IF NOT EXISTS jacek_agents.default;

In [0]:
CATALOG = "jacek_agents"
SCHEMA = "default"

function_info = client.create_python_function(
  func=add_numbers,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True,
)
function_info

In [0]:
import ast
fun_props_dict = ast.literal_eval(function_info.properties)

import pandas as pd
props_df = pd.DataFrame(sorted(list(fun_props_dict.items())), columns=['key', 'value'])
display(props_df)

In [0]:
help(client.create_python_function)

In [0]:
result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.add_numbers",
  parameters={"number_1": 36939.0, "number_2": 8922.4}
)

# client.execute_function returns a string (not a float)
assert float(result.value) == 45861.4


Once tested, add the Unity Catalog function to your agent.

## Add Unity Catalog function to Agent using MCP

[Add Unity Catalog functions to your agent](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#add-unity-catalog-functions-to-your-agent):

There are two approaches to add UC functions to agents:

* Using MCP (recommended)
* Using `UCFunctionToolkit` (to integrate with LangChain, LlamaIndex, OpenAI, Anthropic, and [other third party generative AI frameworks](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration))

The recommended MCP approach provides a simpler integration with automatic tool discovery and built-in authentication support.

Unity Catalog functions executed as AI agent tools require **serverless general compute** (Spark Connect serverless).

### Managed MCP URL for Unity Catalog functions

The managed MCP URL for Unity Catalog functions is: `https://<workspace-hostname>/api/2.0/mcp/functions/{catalog}/{schema}`.

The endpoint above exposes the schema, meaning that all functions created within that schema will be exposed as individual tools of that managed MCP server.

You can optionally specify a specific function by appending `/{function_name}`.

Connect your agent to Unity Catalog functions through MCP.


## Step 1: Create a single-turn LLM call

[Step 1: Create a single-turn LLM call](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api#step-1-create-a-single-turn-llm-call)

### Deploy Agent on Model Serving for GenAI Apps

[Databricks recommends deploying agents on Databricks Apps for full control over agent code, server configuration, and deployment workflow](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent)

Before deploying an agent, you have to register the agent to Unity Catalog.

Registering the agent packages it as a model in Unity Catalog.
As a result, you can use Unity Catalog permissions for authorization for resources in the agent.

In [0]:
# Databricks notebooks already run inside an active asyncio event loop
# DatabricksMCPClient.list_tools() internally calls asyncio.run()
# and fails with RuntimeError: asyncio.run() cannot be called from a running event loop
import nest_asyncio
nest_asyncio.apply()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
import mlflow

workspace_client = WorkspaceClient()
host = workspace_client.config.host

# Connect to the UC functions MCP server
mcp_client = DatabricksMCPClient(
    server_url=f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}",
    workspace_client=workspace_client,
)

# List available tools
mcp_tools = mcp_client.list_tools()
mcp_tools

In [0]:
from databricks_langchain import (
    DatabricksMultiServerMCPClient,
    DatabricksMCPServer
)

databricks_mcp_client = DatabricksMultiServerMCPClient(
    # servers: List of MCPServer or DatabricksMCPServer configurations
    servers = [
        DatabricksMCPServer(
            name="add_numbers",
            url=f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}",
            workspace_client=workspace_client,
        ),
    ]
)

In [0]:
help(DatabricksMultiServerMCPClient)

In [0]:
tools = await databricks_mcp_client.get_tools()
print(f"List of LangChain tools: {tools}")
# databricks_mcp_client.execute(
#     function_name="add_numbers",
#     parameters={"number_1": 36939.0, "number_2": 8922.4}
# )


## Agents

LangChain's [Agents](https://docs.langchain.com/oss/python/langchain/agents):

Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

In [0]:
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
)


LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [0]:
%pip show langchain

In [0]:
%pip show langgraph

In [0]:
from langchain.agents import create_agent

graph = create_agent(
    model=LLM_ENDPOINT_NAME,
    tools=tools,
    system_prompt="You are a helpful assistant",
)

In [0]:
import asyncio
from typing import Annotated, Any, AsyncGenerator, Generator, Optional, Sequence, TypedDict, Union

import mlflow
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
)
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from langchain_core.messages.tool import ToolMessage
import json


nest_asyncio.apply()

LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# TODO: Update with your system prompt
system_prompt = """
You are a helpful assistant that can run Python code.
"""

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools,
    system_prompt: Optional[str] = None,
):
    model = model.bind_tools(tools)  # Bind tools to the model

    # Function to check if agent should continue or finish based on last message
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If function (tool) calls are present, continue; otherwise, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    # Preprocess: optionally prepend a system prompt to the conversation history
    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])

    model_runnable = preprocessor | model  # Chain the preprocessor and the model

    # The function to invoke the model within the workflow
    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(AgentState)  # Create the agent's state machine

    workflow.add_node("agent", RunnableLambda(call_model))  # Agent node (LLM)
    workflow.add_node("tools", ToolNode(tools))  # Tools node

    workflow.set_entry_point("agent")  # Start at agent node
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # If the model requests a tool call, move to tools node
            "end": END,  # Otherwise, end the workflow
        },
    )
    workflow.add_edge("tools", "agent")  # After tools are called, return to agent node

    # Compile and return the tool-calling agent workflow
    return workflow.compile()


# ResponsesAgent class to wrap the compiled agent and make it compatible with Databricks Responses API
class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        self.agent = agent

    # Make a prediction (single-step) for the agent
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    async def _predict_stream_async(
        self,
        request: ResponsesAgentRequest,
    ) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        # Stream events from the agent graph
        async for event in self.agent.astream(
            {"messages": cc_msgs}, stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                # Stream updated messages from the workflow nodes
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        all_messages = []
                        for msg in node_data["messages"]:
                            if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                                msg.content = json.dumps(msg.content)
                            all_messages.append(msg)
                        for item in output_to_responses_items_stream(all_messages):
                            yield item
            elif event[0] == "messages":
                # Stream generated text message chunks
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except:
                    pass

    # Stream predictions for the agent, yielding output as it's generated
    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        agen = self._predict_stream_async(request)

        try:
            loop = asyncio.get_event_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)

        ait = agen.__aiter__()

        while True:
            try:
                item = loop.run_until_complete(ait.__anext__())
            except StopAsyncIteration:
                break
            else:
                yield item


# Initialize the entire agent, including MCP tools and workflow
def initialize_agent():
    """Initialize the agent with MCP tools"""
    # Create MCP tools from the configured servers
    mcp_tools = asyncio.run(databricks_mcp_client.get_tools())

    # Create the agent graph with an LLM, tool set, and system prompt (if given)
    agent = create_tool_calling_agent(llm, mcp_tools, system_prompt)
    return LangGraphResponsesAgent(agent)


In [0]:
import mlflow

mlflow.langchain.autolog()
AGENT = initialize_agent()
mlflow.models.set_model(AGENT)


## Supervisor API
If your agent uses only Databricks-hosted tools and does not need custom logic between tool calls, you can use the [Supervisor API (Beta)](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/supervisor-api) to let Databricks manage the agent loop for you.

The Supervisor API simplifies building custom agents on Databricks.